# 15.14 Greedy and Divide-and-Conquer

**Prerequisites:** 15.13 Dynamic Programming, 15.10 Sorting, 15.12 Recursion  
**Target:** Python 3.12+ (notes flag 3.13/3.14 differences)

### What you'll learn
- **Greedy**: take the best local choice and never reconsider
- 🔴 When greedy is **provably** right - the exchange argument
- 🔴 When it is wrong, and how to spot that in the question
- Interval scheduling, fractional knapsack, **Huffman coding**
- **Divide and conquer**: split, solve, combine
- The **master theorem**, informally - reading the complexity off the recurrence
- Maximum subarray by D&C, and why Kadane beats it
- 🔴 **Greedy vs DP vs D&C** - a decision table
- Interview questions, worked

---

## Greedy: the paradigm

> At each step, take what looks best **right now**, and never reconsider.

```
    sort by some criterion
    for each item:
        if taking it is feasible:
            take it                <- and never undo this
```

No table, no recursion, no backtracking. When it works it is dramatically simpler and faster than DP — usually **O(n log n)** dominated by the sort.

| | Greedy | DP (**15.13**) |
|---|---|---|
| Considers | one choice per step | all choices per step |
| Revisits decisions | never | implicitly, via the table |
| Typical cost | O(n log n) | O(n × states) |
| Correct? | 🔴 **only sometimes — must be proved** | always, if the state is right |

🔴 **That last row is the whole difficulty.** Greedy is easy to write and easy to write *wrongly*. A greedy algorithm that passes your three test cases can still be wrong on the fourth, and there is no warning.

### 🔴 When greedy fails - the evidence from 15.13

**15.13** built three counter-examples. They are worth restating together, because they share one shape:

| Problem | Greedy takes | Optimal | Why greedy loses |
|---|---|---|---|
| Coin change `[1,3,4]`, target 6 | 4+1+1 = **3 coins** | 3+3 = **2 coins** | taking 4 blocks the pair of 3s |
| House robber `[2,7,9,3,1]` | grab 9 first | 2+9+1 = 12 | taking 9 blocks 7 |
| 0/1 knapsack, capacity 8 | best ratio first = 80 | 90 | the best ratio item wastes capacity |

**The common shape:** a choice that looks best now **removes options** that were worth more together. Whenever choices interact like that, greedy needs proof or replacement.

### The test to apply

> Can you construct an input where an early good-looking choice forces a bad ending? If yes, greedy is wrong. If you cannot — and you can *argue* why not — greedy is probably right.

## Proving greedy correct: the exchange argument

The standard technique, and what an interviewer means by *"can you justify that?"*

> **Take any optimal solution. Show you can swap one of its choices for the greedy choice without making it worse. Repeat, and the optimal solution becomes the greedy one — so greedy is optimal too.**

### Worked: interval scheduling

*Given meetings with start and end times, schedule the most non-overlapping ones.*

**Greedy rule: always take the meeting that ends earliest.**

**The argument.** Let `g` be the earliest-ending meeting, and let `O` be some optimal schedule whose first meeting is `o ≠ g`. Since `g` ends no later than `o`, replacing `o` with `g` cannot conflict with anything after it. So `O` with the swap is still optimal and now starts with the greedy choice. Repeat down the schedule. ∎

🔴 **The sort key is the whole algorithm, and the obvious choices are wrong:**

| Sort by | Correct? |
|---|---|
| **earliest end time** | ✅ **yes** |
| earliest start time | 🔴 no — one long early meeting blocks everything |
| shortest duration | 🔴 no — a short meeting can straddle two others |

The cell below runs all three.

In [ ]:
def schedule_by(meetings, key, label):
    """Greedy interval scheduling with a configurable sort key."""
    chosen = []
    finish = float("-inf")
    for start, end in sorted(meetings, key=key):
        if start >= finish:
            chosen.append((start, end))
            finish = end
    return chosen


def best_possible(meetings):
    """Brute force over every subset, to check the greedy answer. 2^n."""
    import itertools

    best = []
    for size in range(len(meetings), 0, -1):
        for combination in itertools.combinations(sorted(meetings), size):
            if all(combination[i][1] <= combination[i + 1][0]
                   for i in range(len(combination) - 1)):
                return list(combination)
    return best


STRATEGIES = [
    ("earliest END", lambda m: m[1]),
    ("earliest start", lambda m: m[0]),
    ("shortest duration", lambda m: m[1] - m[0]),
]

cases = [
    [(1, 3), (2, 5), (4, 7), (1, 8), (5, 9), (8, 10)],
    [(0, 10), (1, 2), (3, 4), (5, 6)],          # defeats earliest-start
    [(1, 5), (4, 6), (5, 10)],                  # defeats shortest-duration
]

for meetings in cases:
    optimal = best_possible(meetings)
    print(f"meetings: {meetings}")
    print(f"  optimal is {len(optimal)}: {optimal}")
    for label, key in STRATEGIES:
        chosen = schedule_by(meetings, key, label)
        flag = "✅" if len(chosen) == len(optimal) else "🔴 WRONG"
        print(f"  {label:<20}{len(chosen)}  {str(chosen):<34}{flag}")
    print()

print("🔴 'Earliest start' loses when one long meeting arrives first.")
print("🔴 'Shortest duration' loses when a short meeting straddles two others.")
print("✅ 'Earliest end' is the one with a proof - and it is the only one")
print("   that is right on all three inputs.")

### Fractional knapsack - where greedy *is* optimal

The same knapsack as **15.13**, with one change: you may take **part** of an item.

That single change makes greedy optimal. Sort by value-per-weight and fill the bag; when the next item does not fit, take the fraction that does.

**Why it works now:** there is no wasted capacity. In the 0/1 version, taking a high-ratio item can leave an unusable gap; here the gap is always filled by a fraction of the next item.

> **This is the sharpest illustration of the greedy/DP boundary in the whole folder.** Identical inputs, one word of difference in the problem statement, and the correct paradigm changes.

In [ ]:
def fractional_knapsack(weights, values, capacity):
    """Greedy by value/weight ratio. PROVABLY optimal when fractions are allowed."""
    order = sorted(range(len(weights)),
                   key=lambda i: values[i] / weights[i], reverse=True)
    total = 0.0
    remaining = capacity
    taken = []
    for i in order:
        if remaining <= 0:
            break
        if weights[i] <= remaining:
            total += values[i]
            remaining -= weights[i]
            taken.append((i, 1.0))
        else:
            fraction = remaining / weights[i]      # take what fits
            total += values[i] * fraction
            taken.append((i, round(fraction, 3)))
            remaining = 0
    return total, taken


def knapsack_01(weights, values, capacity):
    """The 0/1 version from 15.13, for comparison."""
    n = len(weights)
    table = [[0] * (capacity + 1) for _ in range(n + 1)]
    for i in range(1, n + 1):
        for c in range(capacity + 1):
            table[i][c] = table[i - 1][c]
            if weights[i - 1] <= c:
                table[i][c] = max(table[i][c],
                                  table[i - 1][c - weights[i - 1]] + values[i - 1])
    return table[n][capacity]


weights, values, capacity = [3, 4, 5], [30, 50, 60], 8
fractional, taken = fractional_knapsack(weights, values, capacity)
discrete = knapsack_01(weights, values, capacity)

print(f"weights {weights}, values {values}, capacity {capacity}\n")
print(f"  ratios          : {[round(v / w, 2) for w, v in zip(weights, values)]}")
print(f"  FRACTIONAL, greedy : {fractional:.1f}   taking {taken}")
print(f"  0/1, DP            : {discrete}")
print(f"  0/1, greedy        : {sum(values[i] for i, f in taken if f == 1.0)}")
print()
print("  Greedy is OPTIMAL for the fractional version and WRONG for 0/1.")
print("  Same numbers. One word changed in the problem statement.")
print()
print("🔴 In an interview, listen for 'can you take part of an item'.")
print("   It is the difference between a five-line greedy solution and a")
print("   DP table - and between right and wrong.")

### Huffman coding - greedy that ships

Optimal prefix-free compression, and it is a greedy algorithm inside every ZIP, JPEG and MP3 file.

**The idea:** frequent symbols get short codes, rare symbols long ones.

```
   repeatedly: take the TWO least frequent nodes,
               merge them under a new parent,
               put the parent back
```

That is a min-heap loop (**15.8**) building a binary tree (**15.7**) bottom-up. The greedy choice — always merge the two rarest — is provably optimal by an exchange argument.

**Prefix-free** means no code is a prefix of another, so the stream decodes unambiguously with no separators. That falls out of putting every symbol at a **leaf**.

🔴 Huffman is optimal *among prefix-free codes assigning whole bits per symbol*. Arithmetic coding does better by escaping that constraint — worth knowing the limitation, since "optimal" is doing careful work in that sentence.

In [ ]:
import heapq
from collections import Counter


def huffman_codes(text):
    """Build optimal prefix-free codes. O(n + k log k) for k distinct symbols."""
    frequencies = Counter(text)
    if len(frequencies) == 1:                    # 🔴 single-symbol edge case
        return {next(iter(frequencies)): "0"}, frequencies

    # (frequency, tiebreaker, node) - the counter avoids comparing dicts (15.8)
    counter = iter(range(len(frequencies) * 2))
    heap = [(freq, next(counter), symbol) for symbol, freq in frequencies.items()]
    heapq.heapify(heap)

    while len(heap) > 1:
        freq_a, _, left = heapq.heappop(heap)    # the two RAREST
        freq_b, _, right = heapq.heappop(heap)
        heapq.heappush(heap, (freq_a + freq_b, next(counter), (left, right)))

    codes = {}

    def walk(node, prefix):
        if isinstance(node, tuple):
            walk(node[0], prefix + "0")
            walk(node[1], prefix + "1")
        else:
            codes[node] = prefix

    walk(heap[0][2], "")
    return codes, frequencies


text = "abracadabra"
codes, frequencies = huffman_codes(text)

print(f"text: {text!r}\n")
print(f"{'symbol':>8}{'frequency':>11}{'code':>10}{'bits':>7}")
print("-" * 36)
for symbol in sorted(codes, key=lambda s: (-frequencies[s], s)):
    print(f"{symbol!r:>8}{frequencies[symbol]:>11}{codes[symbol]:>10}"
          f"{len(codes[symbol]):>7}")

encoded = "".join(codes[ch] for ch in text)
fixed_width = len(text) * 8
print(f"\n  fixed 8-bit encoding : {fixed_width} bits")
print(f"  Huffman encoding     : {len(encoded)} bits")
print(f"  saving               : {100 * (1 - len(encoded) / fixed_width):.0f}%")

# prefix-free means it decodes with no separators
reverse = {code: symbol for symbol, code in codes.items()}
decoded, buffer = [], ""
for bit in encoded:
    buffer += bit
    if buffer in reverse:
        decoded.append(reverse[buffer])
        buffer = ""
print(f"\n  decoded correctly: {''.join(decoded) == text}")
print("  ^ no delimiters needed - no code is a prefix of another, which")
print("    is guaranteed because every symbol sits at a LEAF.")

print(f"\n  the most frequent symbol {max(frequencies, key=frequencies.get)!r} "
      f"got the shortest code. That is the greedy rule paying off.")

---

# Divide and conquer

Three steps:

```
   1. DIVIDE    break the problem into smaller instances of ITSELF
   2. CONQUER   solve those recursively
   3. COMBINE   assemble their answers
```

You have used it repeatedly already:

| Algorithm | Divide | Combine | Cost |
|---|---|---|---|
| **Merge sort** (**15.10**) | halve | merge, O(n) | O(n log n) |
| **Quicksort** (**15.10**) | partition, O(n) | nothing | O(n log n) avg |
| **Binary search** (**15.11**) | halve | nothing | O(log n) |
| **Quickselect** (**15.11**) | partition | nothing | O(n) avg |
| Tree traversal (**15.7**) | left, right | combine results | O(n) |

### 🔴 How it differs from DP

> Divide and conquer splits into **disjoint** subproblems. DP applies when they **overlap**.

Merge sort's two halves share nothing, so caching would never hit — memoising it wastes memory for zero gain (**15.13**). Fibonacci's two branches share almost everything, which is why caching is transformative.

## The master theorem, informally

Most divide-and-conquer algorithms fit one recurrence:

```
    T(n) = a·T(n/b) + f(n)
           ^   ^^^^   ^^^^
           |    |      work to split and combine
           |    size of each subproblem
           how many subproblems
```

Compare the **recursion work** with the **combine work** and the larger one wins:

| Case | Condition | Result |
|---|---|---|
| 1 | combining is cheap | O(n^log_b(a)) — the leaves dominate |
| 2 | they are balanced | O(n^log_b(a) · log n) |
| 3 | combining is expensive | O(f(n)) — the root dominates |

### Applied

| Algorithm | a | b | f(n) | Result |
|---|---|---|---|---|
| Merge sort | 2 | 2 | O(n) | balanced → **O(n log n)** |
| Binary search | 1 | 2 | O(1) | balanced → **O(log n)** |
| Quickselect (avg) | 1 | 2 | O(n) | root dominates → **O(n)** |
| Karatsuba multiply | 3 | 2 | O(n) | leaves dominate → **O(n^1.585)** |

> **You do not need the formal statement.** You need to answer *how many subproblems, how much smaller, and how much work to combine* — and the answer usually falls out.

**Karatsuba** is the memorable one: multiplying two n-digit numbers naively is O(n²), but it can be done with **three** half-size multiplications instead of four, giving O(n^1.585). Python uses it internally for large integers.

In [ ]:
import math
import random
import time


def max_subarray_divide(data):
    """Maximum subarray sum by divide and conquer. O(n log n).

    The best subarray is entirely left, entirely right, or CROSSES the
    middle - and the crossing case is what the combine step computes.
    """
    def solve(lo, hi):
        if lo == hi:
            return data[lo]
        mid = (lo + hi) // 2

        left_best = solve(lo, mid)                  # CONQUER
        right_best = solve(mid + 1, hi)

        # COMBINE: the best run ending at mid, plus the best starting after
        running = 0
        left_edge = float("-inf")
        for i in range(mid, lo - 1, -1):
            running += data[i]
            left_edge = max(left_edge, running)
        running = 0
        right_edge = float("-inf")
        for i in range(mid + 1, hi + 1):
            running += data[i]
            right_edge = max(right_edge, running)

        return max(left_best, right_best, left_edge + right_edge)

    return solve(0, len(data) - 1) if data else 0


def max_subarray_kadane(data):
    """Kadane from 15.3. O(n) - and it is a one-dimensional DP."""
    if not data:
        return 0
    best = current = data[0]
    for value in data[1:]:
        current = max(value, current + value)
        best = max(best, current)
    return best


rng = random.Random(15)
for sample in ([-2, 1, -3, 4, -1, 2, 1, -5, 4], [-3, -1, -2], [5], [1, 2, 3]):
    dc = max_subarray_divide(sample)
    kadane = max_subarray_kadane(sample)
    print(f"  {str(sample):<32} D&C {dc:>3}   Kadane {kadane:>3}   "
          f"agree: {dc == kadane}")

big = [rng.randint(-100, 100) for _ in range(60_000)]
started = time.perf_counter()
dc_result = max_subarray_divide(big)
dc_time = time.perf_counter() - started
started = time.perf_counter()
kadane_result = max_subarray_kadane(big)
kadane_time = time.perf_counter() - started

print(f"\n  n = {len(big):,}, same answer: {dc_result == kadane_result}")
print(f"    divide and conquer  O(n log n)  {dc_time * 1000:8.1f} ms")
print(f"    Kadane              O(n)        {kadane_time * 1000:8.1f} ms   "
      f"{dc_time / kadane_time:.0f}x faster")
print("\n  🔴 D&C is not automatically the best tool. Here a one-pass DP")
print("     beats it - and is shorter. Reach for D&C when the problem")
print("     genuinely splits, not because splitting feels clever.")

In [ ]:
# Karatsuba - the memorable divide-and-conquer win
def karatsuba(x, y):
    """Multiply with THREE half-size products instead of four. O(n^1.585).

        x = a*10^m + b        y = c*10^m + d
        xy = ac*10^2m + (ad + bc)*10^m + bd

    Naively that is four products. Karatsuba computes ad+bc as
        (a+b)(c+d) - ac - bd
    reusing ac and bd - so three products suffice.
    """
    if x < 10 or y < 10:
        return x * y

    m = max(len(str(x)), len(str(y))) // 2
    power = 10 ** m
    a, b = divmod(x, power)
    c, d = divmod(y, power)

    ac = karatsuba(a, c)                       # 1
    bd = karatsuba(b, d)                       # 2
    cross = karatsuba(a + b, c + d) - ac - bd  # 3 - not 4

    return ac * power * power + cross * power + bd


rng = random.Random(15)

# 🔴 An earlier version of this check was vacuous: it used
#     all(... for _ in range(0)) or all(... for _ in range(500))
# and `all()` over an EMPTY generator is True, so the `or` short-
# circuited and the real comparison never ran. It printed True.
checked = 0
mismatches = 0
for _ in range(500):
    a = rng.randint(0, 10 ** 12)
    b = rng.randint(0, 10 ** 12)
    checked += 1
    if karatsuba(a, b) != a * b:
        mismatches += 1

print(f"karatsuba checked against * on {checked} random pairs: "
      f"{mismatches} mismatches")

a = 3141592653589793238462643383279502884197169399375105820974944592
b = 2718281828459045235360287471352662497757247093699959574966967627
print(f"\n  on two 64-digit numbers: {karatsuba(a, b) == a * b}")
print(f"  product has {len(str(karatsuba(a, b)))} digits")

print(f"\n  the recurrence: T(n) = 3*T(n/2) + O(n)")
print(f"  log2(3) = {math.log2(3):.3f}, so O(n^{math.log2(3):.3f}) beats O(n^2)")
print("\n  Python uses Karatsuba internally for large integers - which is")
print("  why the built-in * will still beat this Python implementation.")

## 🔴 Choosing the paradigm

| | **Greedy** | **Divide & conquer** | **Dynamic programming** |
|---|---|---|---|
| Subproblems | one path only | **disjoint** | **overlapping** |
| Revisits choices | never | n/a | yes, via the table |
| Typical cost | O(n log n) | O(n log n) | O(n × states) |
| Memory | O(1) | O(log n) stack | O(states) |
| Correctness | 🔴 must be **proved** | structural | guaranteed if the state is right |
| Write it when | a local choice is provably safe | the problem splits cleanly | choices interact |

### The decision procedure

```
   1. Can I split into INDEPENDENT subproblems?     -> divide and conquer
   2. Do subproblems REPEAT?                        -> dynamic programming
   3. Is one local choice provably always safe?     -> greedy
   4. None of the above?                            -> backtracking (15.12)
```

### The honest interview advice

> **Try greedy first — and then try to break it.** Spend one minute constructing a counter-example. If you find one, you now understand why DP is needed and you can say so. If you cannot, sketch the exchange argument.

Saying *"greedy would take X, but consider this input where that blocks Y — so I need DP here"* demonstrates exactly the judgement being assessed.

## Interview questions

**1. What is a greedy algorithm, and when is it valid?**
> Take the locally best choice and never reconsider. Valid only when it can be proved — usually by an exchange argument. Give a counter-example where it fails.

**2. Interval scheduling / meeting rooms.** *(above)*
> Sort by **earliest end time**. Explain why earliest-start and shortest-duration both fail. The 'how many rooms' variant is a heap (**15.8**).

**3. Fractional vs 0/1 knapsack.** *(above)*
> Greedy by ratio is optimal for fractional; 0/1 needs DP. Listen for whether items can be split.

**4. Jump game — can you reach the end?**
> Greedy: track the furthest index reachable so far. O(n). The minimum-jumps variant is also greedy, with a level-by-level scan.

**5. Huffman coding.** *(above)*
> Repeatedly merge the two least frequent nodes with a min-heap. Optimal prefix-free code; note the whole-bits caveat.

**6. Gas station — can you complete the circuit?**
> Greedy: if the running tank goes negative, no start before this point works, so restart from the next station. O(n), and the argument is the interesting part.

**7. What is divide and conquer? How does it differ from DP?**
> Split, solve recursively, combine. Subproblems are disjoint; DP is for overlapping ones.

**8. Maximum subarray, divide and conquer.** *(above)*
> Left, right, or crossing. O(n log n) — then note Kadane does it in O(n), and that Kadane is DP.

**9. Merge k sorted lists.**
> D&C: merge pairwise, halving k each round — O(N log k), the same as the heap approach (**15.8**).

**10. Multiply two large numbers faster than O(n²).**
> Karatsuba: three half-size products instead of four, O(n^1.585).

In [ ]:
# Questions 4 and 6 - two greedy proofs worth being able to state.
def can_jump(nums):
    """Greedy: track the furthest reachable index. O(n) time, O(1) space."""
    furthest = 0
    for index, jump in enumerate(nums):
        if index > furthest:
            return False              # this index is unreachable
        furthest = max(furthest, index + jump)
    return True


def min_jumps(nums):
    """Fewest jumps to the end. Greedy, level by level - it is BFS (15.9)."""
    if len(nums) <= 1:
        return 0
    jumps = 0
    current_end = furthest = 0
    for index in range(len(nums) - 1):
        furthest = max(furthest, index + nums[index])
        if index == current_end:      # the end of this 'level'
            jumps += 1
            current_end = furthest
    return jumps


def can_complete_circuit(gas, cost):
    """Gas station. If the tank goes negative at i, no start in 0..i works.

    That claim is the whole proof - every earlier start arrives at i with
    at most as much fuel, so it cannot do better.
    """
    if sum(gas) < sum(cost):
        return -1                     # not enough fuel in total
    start = tank = 0
    for i in range(len(gas)):
        tank += gas[i] - cost[i]
        if tank < 0:
            start = i + 1             # restart from the next station
            tank = 0
    return start


print("jump game:")
for nums in ([2, 3, 1, 1, 4], [3, 2, 1, 0, 4], [0], [2, 0, 0]):
    reachable = can_jump(nums)
    detail = f"  fewest jumps: {min_jumps(nums)}" if reachable else ""
    print(f"  {str(nums):<18} reachable: {str(reachable):<6}{detail}")

print("\ngas station:")
for gas, cost in (([1, 2, 3, 4, 5], [3, 4, 5, 1, 2]),
                 ([2, 3, 4], [3, 4, 3]),
                 ([5, 1, 2, 3, 4], [4, 4, 1, 5, 1])):
    start = can_complete_circuit(gas, cost)
    print(f"  gas {gas}, cost {cost} -> start at {start}")

# verify the gas-station answer by brute force
def brute_circuit(gas, cost):
    n = len(gas)
    for start in range(n):
        tank = 0
        for step in range(n):
            i = (start + step) % n
            tank += gas[i] - cost[i]
            if tank < 0:
                break
        else:
            return start
    return -1


rng = random.Random(15)
agree = True
for _ in range(300):
    n = rng.randint(1, 7)
    gas = [rng.randint(0, 5) for _ in range(n)]
    cost = [rng.randint(0, 5) for _ in range(n)]
    if can_complete_circuit(gas, cost) != brute_circuit(gas, cost):
        agree = False
        break
print(f"\n  greedy matches brute force on 300 random circuits: {agree}")
print("  ^ this is how you gain confidence in a greedy algorithm you")
print("    cannot fully prove: test it against exhaustive search.")

---

## Common Mistakes & Pitfalls

1. 🔴 **Using greedy without justifying it.** It is easy to write and easy to write wrongly - and it fails silently on inputs you did not try.
2. 🔴 **Sorting by the wrong key in interval scheduling.** Earliest *end*, not earliest start and not shortest duration.
3. 🔴 **Applying fractional-knapsack greedy to 0/1.** The one word 'fractional' decides which paradigm is correct.
4. **Confusing D&C with DP.** Disjoint subproblems versus overlapping ones - caching merge sort gains nothing.
5. **Reaching for divide and conquer when a linear pass exists.** Kadane beats the O(n log n) D&C maximum subarray.
6. **Forgetting the single-symbol case in Huffman.** One distinct symbol builds no tree and produces an empty code.
7. **Comparing payloads in a heap without a tiebreaker** when building Huffman trees (**15.8**).
8. **Claiming Huffman is 'optimal' without qualification.** It is optimal among prefix-free whole-bit codes; arithmetic coding does better.
9. **Assuming a greedy proof exists because your tests pass.** Test against brute force on small inputs.

## Best Practices

- Try greedy first, then spend a minute trying to break it.
- Learn the exchange argument - it is what 'justify that' means.
- Verify greedy algorithms against exhaustive search on small random inputs.
- Ask whether subproblems overlap: that single question separates D&C from DP.
- Identify a, b and f(n) to read a divide-and-conquer complexity off the recurrence.
- Prefer the simplest paradigm that is provably correct - greedy over DP when both work.
- State the sort key and *why* it is the right one when presenting a greedy solution.
- Say out loud when greedy fails and why - it demonstrates the judgement being assessed.

## Practice Exercises

Try these before moving on.

1. 🔴 Construct an input where 'earliest start time' beats 'earliest end time' for interval scheduling - or argue convincingly that none exists.
2. Implement 'minimum number of platforms' for a railway timetable. Which greedy rule, and which data structure (**15.8**)?
3. Prove the gas-station greedy rule: why can no start before a failure point work?
4. Implement Huffman decoding directly from the tree rather than from a code dictionary. Which is faster, and why?
5. Solve 'assign cookies to children' greedily, and state the exchange argument.
6. Implement Strassen's matrix multiplication - seven multiplications instead of eight - and identify a, b and f(n).
7. Compare merging k sorted lists by pairwise D&C against the heap method (**15.8**) for k = 64. Are the complexities really the same?
8. 🔴 Take a greedy solution you believe is correct and write a brute-force checker plus a random input generator. Run 10,000 cases. This is the most useful habit in this notebook.